# Out-of-Distribution Testing 

In [ ]:
import numpy as np
from matplotlib import pyplot as plt 
import time as time
import sys
import platform
import psutil

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

from captum.attr import Saliency

from sklearn.model_selection import train_test_split

sys.path.append('..')

from prox_op import prox_op
from data_fcns import generate_raw_data
from data_fcns import vanilla_preprocess
from data_fcns import compute_features

In [ ]:
# python version

print(sys.version)

In [ ]:
# get CPU info

print(platform.processor())
print(platform.machine())
print(platform.version())
print(platform.platform())
print(platform.uname())
print(platform.system())
print(str(round(psutil.virtual_memory().total / (1024.0 **3)))+" GB")


In [ ]:
# get GPU info

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.device_count())
    print(torch.cuda.current_device())
    print(torch.cuda.device(0))
    print(torch.cuda.get_device_name(0))

#### Data Prep

##### Create data

In [ ]:
nn_type = "feature"  # vanilla or feature
data_dist = "unif"   # norm, unif, or both
unif_min = 0
unif_max = 1
min_len = 1000
max_len = 2000
num_vec = 10000
seed = 1

X, lengths, alphas, taus = generate_raw_data(data_dist, min_len, max_len, num_vec, unif_min, unif_max, seed)

In [ ]:
print(lengths.shape)
print(taus.shape)
print(alphas.shape)
print(X.shape)

##### preprocess data

In [ ]:
if nn_type == "vanilla":

    num_moments = 0 # needed for the NN selection cell
    M, yhat, zero_idx = vanilla_preprocess(X, lengths, alphas, taus, seed)
    
else:  # features NN
    num_moments = 10
    M, yhat, mus, zero_idx = compute_features(X, lengths, alphas, taus, num_moments)


In [ ]:
# remove any observations from dataset that have tau = 0 

print(sum(zero_idx))

if sum(zero_idx) > 0:
    M = M[~zero_idx,:]
    yhat = yhat[~zero_idx]
    mus = mus[~zero_idx]
    alphas = alphas[~zero_idx]
    taus = taus[~zero_idx]
    
if data_dist == "norm":
    num_norm_vec = M.shape[0]
elif data_dist == "unif":  
    num_norm_vec = 0
else: # both
    num_norm_vec = int(num_vec/2) - sum(np.where(zero_idx)[0] < int(num_vec/2))
    
print(num_norm_vec)

### NN

In [ ]:
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
class TauDataset(Dataset):
    def __init__(self, X, y, num_norm_vec):
        
        d = np.zeros(len(y))
        d[:num_norm_vec] = 1  # Gaussian - 1, Uniform - 0
          
        self.features = torch.Tensor(X)
        self.labels = torch.Tensor(y) 
        self.dist = torch.Tensor(d)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx,:], self.labels[idx], self.dist[idx], idx

In [ ]:
# create dataset
dataset = TauDataset(M,yhat,num_norm_vec)

print(dataset[0])
print(len(dataset))

In [ ]:
if data_dist == "both":

    # train-test split: 80-20, stratified sampling on distribution type
    # Gaussian: 1st half of indices, Uniform: 2nd half of indices 

    train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.20, random_state=0, 
                                       shuffle=True, stratify=dataset.dist)
    
else: # unif or norm data
    
    # train-test split: 80-20
    train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.20, random_state=0, shuffle=True)


In [ ]:
sum(dataset.dist[train_idx])

In [ ]:
sum(dataset.dist[test_idx])

In [ ]:
train_data = Subset(dataset, train_idx)
test_data = Subset(dataset, test_idx)

In [ ]:
# set up dataset iterator

train_batch_size = 32
test_batch_size = len(test_data)

train_loader = DataLoader(dataset=train_data, batch_size=train_batch_size, shuffle=True) 
test_loader = DataLoader(dataset=test_data, batch_size=test_batch_size, shuffle=True)

In [ ]:
print(len(train_loader))
print(len(test_loader))

#### Build NN

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

In [ ]:
# first layer number of inputs
print(dataset[0][0].size())
layer1_size = M[0].shape[0]
layer1_size

In [ ]:
# set NN based on layer1_size

if layer1_size == (num_moments+3):  # features NN
    
    class NeuralNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear_relu_stack = nn.Sequential(
                nn.Linear(num_moments+3, 25),  
                nn.ReLU(),    
                nn.Linear(25, 10),
                nn.ReLU(),
                nn.Linear(10, 1)
            )

        def forward(self, x):
            tau = self.linear_relu_stack(x) 
            return tau
        
elif (layer1_size == 2000) or (layer1_size == 100000):   # vanilla NN
    
    class NeuralNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear_relu_stack = nn.Sequential(
                nn.Linear(layer1_size, 200),   
                nn.ReLU(),  
                nn.Linear(200, 100),
                nn.ReLU(),
                nn.Linear(100, 50), 
                nn.ReLU(),
                nn.Linear(50, 1) 
            )

        def forward(self, x):
            tau = self.linear_relu_stack(x)
            return tau
    
else:
    pass

model = NeuralNetwork().to(device)
print(model)

In [ ]:
# load model

model_path = "/home/kjl5t/L_inf_prox_op/src/models/"

#model = NeuralNetwork()
model.load_state_dict(torch.load(model_path + "features/gaussian/len_1000_2000/epoch_4421_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features/gaussian/len_1000_100000/epoch_4695_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features/uniform_0_1/len_1000_2000/epoch_4897_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features/uniform_0_1/len_1000_100000/epoch_4928_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features/both/len_1000_2000/epoch_4570_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features/both/len_1000_100000/epoch_4792_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features_5m/both/len_1000_2000/epoch_4370_nn.pt"))
#model.load_state_dict(torch.load(model_path + "features_5m/both/len_1000_100000/epoch_4773_nn.pt"))

# density models
#model.load_state_dict(torch.load(model_path + "density/unif_0_10/len_1000_2000/epoch_3788_nn.pt"))
#model.load_state_dict(torch.load(model_path + "density/unif_0_10/len_1000_100000/epoch_4390_nn.pt"))
#model.load_state_dict(torch.load(model_path + "density/unif_0_20/len_1000_100000/epoch_3779_nn.pt"))
#model.load_state_dict(torch.load(model_path + "density/unif_0_20/len_1000_2000/epoch_4551_nn.pt"))

model.eval()

In [ ]:
# Testing on CPU or GPU - use appropriate timing commands!

def test(dataloader, model, loss_fn):
    
    #size = len(dataloader.dataset)
    
    # GPU timers
    #start_time = torch.cuda.Event(enable_timing=True)
    #end_time = torch.cuda.Event(enable_timing=True)
    
    num_batches = len(dataloader)
   
    model.eval()
    test_loss_all = 0
    test_loss_norm = 0
    test_loss_unif = 0
    test_loss_og = 0
    epoch_time = 0
    with torch.no_grad():
        for X, y, dist, idx in dataloader:
            X, y, dist, idx = X.to(device), y.to(device), dist.to(device), idx.to(device)
            
            # CPU time
            t1 = time.perf_counter()
            # GPU time
            #start_time.record()
            
            pred = model(X)
            
            # CPU time
            t2 = time.perf_counter()
            epoch_time += t2-t1
            # GPU time
            #end_time.record()
            #torch.cuda.synchronize()
            #epoch_time += (start_time.elapsed_time(end_time))/1000  # time unit is milliseconds
            
            test_loss_all += loss_fn(pred, y.unsqueeze(1)).item()
            test_loss_norm += loss_fn(pred[dist == 1], y[dist == 1].unsqueeze(1)).item() 
            test_loss_unif += loss_fn(pred[dist == 0], y[dist == 0].unsqueeze(1)).item()
    
            # additional code - loss on original tau.  If use GPU - must send pred to CPU

            if device == "cuda":
                pred_og_tau = pred.squeeze().cpu().numpy()
                idx = idx.cpu()
            else:
                pred_og_tau = pred.squeeze().numpy()
        
            if nn_type == "feature":

                pred_og_tau = np.add(pred_og_tau, mus[idx])
                pred_og_tau = np.multiply(alphas[idx], pred_og_tau)
                test_loss_og += loss_fn(torch.Tensor(pred_og_tau), torch.Tensor(taus[idx])).item()     
                    
            else:  # vanilla NN
                
                pred_og_tau = np.multiply(alphas[idx], pred_og_tau)
                test_loss_og += loss_fn(torch.Tensor(pred_og_tau), torch.Tensor(taus[idx])).item()             
                
    test_loss_all /= num_batches
    test_loss_norm /= num_batches
    test_loss_unif /= num_batches
    test_loss_og /= num_batches
    
    print(f"Epoch test time: {epoch_time} seconds\n")
    print(f"Test Error: \n Avg loss (per obs): {test_loss_all:.2e} \n") 
    print(f"Gaussian Test Error: \n Avg loss (per obs): {test_loss_norm:.2e} \n")
    print(f"Uniform Test Error: \n Avg loss (per obs): {test_loss_unif:.2e} \n")
    print(f"Original Tau Test Error: \n Avg loss (per obs): {test_loss_og:.2e} \n")   

    return test_loss_all, test_loss_norm, test_loss_unif, test_loss_og, epoch_time, pred_og_tau, taus[idx] 

In [ ]:
len(test_loader.dataset)

In [ ]:
loss_fn = nn.MSELoss()

In [ ]:
avg_loss_per_obs_test, avg_loss_per_obs_test_norm, avg_loss_per_obs_test_unif, avg_loss_per_obs_test_og, test_time, pred_tau_og, tau_og = test(test_loader, model, loss_fn)